# 导入包

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch
import sys, os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sparse
import matplotlib.pyplot as plt
from scipy.sparse import issparse

In [ ]:
from SpatialGlue.preprocess import fix_seed, pca, construct_neighbor_graph

# 小脑

In [ ]:
# Cell 1：导入、路径和参数
# ============================================================

import os
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch

from scipy import sparse
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# ------------------------- 样本选择 -------------------------

# 只改这一行："G24-2" 或 "G25-6"
SAMPLE_KEY = "G24-2"

BASE_DIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie"
)

SAMPLE_CONFIG = {
    "G24-2": {
        "sample_name": "G24-2_Cerebellum",
        "gene_path": (
            BASE_DIR
            / "20260210_2/0423_repaired_data/G24-2_repaired.h5ad"
        ),
        "psi_dir": (
            BASE_DIR
            / "20260427_AS_analysis/Fig4_cluster_psi/results"
            / "G24-2_Cerebellum"
        ),
    },
    "G25-6": {
        "sample_name": "G25-6_Kidney",
        "gene_path": (
            BASE_DIR
            / "20260210_2/0423_repaired_data/G25-6_repaired.h5ad"
        ),
        "psi_dir": (
            BASE_DIR
            / "20260427_AS_analysis/Fig4_cluster_psi/results"
            / "G25-6_Kidney"
        ),
    },
}

if SAMPLE_KEY not in SAMPLE_CONFIG:
    raise KeyError(
        f"未知SAMPLE_KEY：{SAMPLE_KEY}"
    )

cfg = SAMPLE_CONFIG[SAMPLE_KEY]

SAMPLE_NAME = cfg["sample_name"]
GENE_PATH = cfg["gene_path"]
INCLUSION_PATH = cfg["psi_dir"] / "inc.h5ad"
SKIPPING_PATH = cfg["psi_dir"] / "ski.h5ad"
OUT_DIR = cfg["psi_dir"] / "SpatialGlue_gene_PSI"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------- PSI构建参数 -------------------------

IN_TISSUE_ONLY = True

# 优先读取layers["counts"]；不存在时读取X
COUNT_LAYER = "counts"

# spot-event至少需要多少个I+S计数才认为PSI可观测
PSI_MIN_TOTAL = 1

# 是否沿用旧脚本的rMATS显著事件筛选
# 无监督聚类建议False，避免把已有分组信息带入聚类
USE_RMATS_SIG_FILTER = False
FDR_THRESH = 0.05
DPSI_THRESH = 0.05


# ------------------------- 预处理参数 -------------------------

GENE_MIN_SPOTS = 5
N_TOP_GENES = 3000

# 一个事件至少在1%的spot中有I+S计数
PSI_MIN_OBS_FRAC = 0.01
N_TOP_PSI = 3000
N_PCS = 50


# ------------------------- SpatialGlue参数 -------------------------

RANDOM_SEED = 2022
DATA_TYPE = "10x"
SPATIAL_NEIGHBORS = 3

EPOCHS = 200
LEARNING_RATE = 1e-4
LATENT_DIM = 64

# [gene重建, PSI重建, gene对应, PSI对应]
# 等权设置避免预先偏向PSI，适合评价PSI是否提供额外信息
WEIGHT_FACTORS = [1, 1, 1, 1]

CLUSTER_METHOD = "leiden"
N_CLUSTERS = 15
CLUSTER_KEY = "SpatialGlue_cluster"


# ------------------------- 环境和绘图 -------------------------

R_HOME = "/histor/zhao/lupeng/micromamba/envs/SpatialGlue/lib/R"
os.environ["R_HOME"] = R_HOME

from SpatialGlue.preprocess import (
    construct_neighbor_graph,
    fix_seed,
    pca,
)
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue
from SpatialGlue.utils import clustering

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
sc.settings._vector_friendly = True

fix_seed(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", device)
print("Sample:", SAMPLE_NAME)
print("Gene:", GENE_PATH)
print("Inclusion:", INCLUSION_PATH)
print("Skipping:", SKIPPING_PATH)
print("Output:", OUT_DIR)


In [ ]:
# Cell 2：读取数据并对齐spot和事件
# ============================================================


def get_count_matrix(adata, layer="counts"):
    """优先读取原始counts层；不存在时使用X。"""
    if layer is not None and layer in adata.layers:
        matrix = adata.layers[layer]
        source = f'layers["{layer}"]'
    else:
        matrix = adata.X
        source = "X"

    matrix = sparse.csr_matrix(
        matrix,
        dtype=np.float32,
    )

    if matrix.nnz > 0 and matrix.data.min() < 0:
        raise ValueError("计数矩阵不能包含负值")

    return matrix, source


gene = sc.read_h5ad(GENE_PATH)
inc = ad.read_h5ad(INCLUSION_PATH)
ski = ad.read_h5ad(SKIPPING_PATH)

print("原始矩阵")
print("Gene:      ", gene.shape)
print("Inclusion: ", inc.shape)
print("Skipping:  ", ski.shape)


# 只保留组织内spot
if IN_TISSUE_ONLY and "in_tissue" in gene.obs:
    keep = (
        gene.obs["in_tissue"]
        .astype(str)
        .str.lower()
        .isin(["1", "true"])
    )
    gene = gene[keep].copy()


# 按Gene矩阵顺序对齐spot
common_spots = gene.obs_names[
    gene.obs_names.isin(inc.obs_names)
    & gene.obs_names.isin(ski.obs_names)
]

if len(common_spots) == 0:
    raise ValueError("三个矩阵没有共同spot")

gene = gene[common_spots].copy()
inc = inc[common_spots].copy()
ski = ski[common_spots].copy()


# 按Inclusion矩阵顺序对齐事件
common_events = inc.var_names[
    inc.var_names.isin(ski.var_names)
]

if len(common_events) == 0:
    raise ValueError("Inclusion与Skipping没有共同事件")

inc = inc[:, common_events].copy()
ski = ski[:, common_events].copy()

assert np.array_equal(inc.obs_names, ski.obs_names)
assert np.array_equal(inc.var_names, ski.var_names)

if "spatial" not in gene.obsm:
    raise KeyError('Gene矩阵缺少obsm["spatial"]')

inc_counts, inc_source = get_count_matrix(inc, COUNT_LAYER)
ski_counts, ski_source = get_count_matrix(ski, COUNT_LAYER)

print("\n对齐后")
print("Spots: ", len(common_spots))
print("Events:", len(common_events))
print("Inclusion counts来源:", inc_source)
print("Skipping counts来源: ", ski_source)


## 构建psi矩阵

In [ ]:
# Cell 3：构建并保存PSI矩阵
# ============================================================

event_var = inc.var.copy()


# 可选：沿用旧脚本的FDR和dPSI筛选
if USE_RMATS_SIG_FILTER:
    required = ["FDR", "dPSI"]
    missing = [x for x in required if x not in event_var]
    if missing:
        raise KeyError(f"事件信息缺少列：{missing}")

    fdr = pd.to_numeric(
        event_var["FDR"],
        errors="coerce",
    )
    dpsi = pd.to_numeric(
        event_var["dPSI"],
        errors="coerce",
    )

    keep_event = (
        fdr.notna()
        & (fdr <= FDR_THRESH)
        & dpsi.notna()
        & (dpsi.abs() > DPSI_THRESH)
    ).to_numpy()

    inc_counts = inc_counts[:, keep_event]
    ski_counts = ski_counts[:, keep_event]
    event_var = event_var.iloc[keep_event].copy()

    print("rMATS显著筛选后事件数:", keep_event.sum())


# 必须使用原始I/S计数计算，不对两个矩阵分别normalize_total
total_counts = (
    inc_counts + ski_counts
).tocsr()

# observed=1：该spot-event的I+S达到最低计数
observed = total_counts.copy()
observed.data = (
    observed.data >= PSI_MIN_TOTAL
).astype(np.uint8)
observed.eliminate_zeros()

# 稀疏计算 PSI = I/(I+S)
inverse_total = total_counts.copy().astype(np.float32)
valid_data = inverse_total.data >= PSI_MIN_TOTAL
inverse_total.data = np.where(
    valid_data,
    1.0 / inverse_total.data,
    0.0,
).astype(np.float32)
inverse_total.eliminate_zeros()

psi_matrix = (
    inc_counts.multiply(inverse_total)
).tocsr().astype(np.float32)
psi_matrix.eliminate_zeros()

if psi_matrix.nnz > 0:
    if psi_matrix.data.min() < -1e-7 or psi_matrix.data.max() > 1 + 1e-7:
        raise ValueError("计算得到的PSI不在0–1范围")


psi = ad.AnnData(
    X=psi_matrix,
    obs=gene.obs.copy(),
    var=event_var.copy(),
)

psi.layers["inclusion_counts"] = inc_counts.astype(np.float32)
psi.layers["skipping_counts"] = ski_counts.astype(np.float32)
psi.layers["total_counts"] = total_counts.astype(np.float32)
psi.layers["observed"] = observed

psi.obsm["spatial"] = gene.obsm["spatial"].copy()
if "spatial" in gene.uns:
    psi.uns["spatial"] = gene.uns["spatial"].copy()

psi.uns["psi_definition"] = {
    "formula": "inclusion_counts / (inclusion_counts + skipping_counts)",
    "count_source_inclusion": inc_source,
    "count_source_skipping": ski_source,
    "min_total_per_spot_event": PSI_MIN_TOTAL,
    "missing_definition": "I + S < min_total",
    "missing_not_equal_zero": True,
    "rmats_significant_filter": USE_RMATS_SIG_FILTER,
}

PSI_PATH = OUT_DIR / f"{SAMPLE_NAME}_PSI.h5ad"
psi.write_h5ad(PSI_PATH, compression="gzip")

n_all = psi.n_obs * psi.n_vars
n_observed = observed.nnz
n_true_zero = observed.nnz - psi_matrix.nnz

print("PSI矩阵构建完成")
print("Shape:            ", psi.shape)
print("Observed比例:     ", round(n_observed / n_all, 4))
print("缺失比例:         ", round(1 - n_observed / n_all, 4))
print("真实PSI=0位置数:  ", f"{n_true_zero:,}")
print("保存路径:         ", PSI_PATH)


In [ ]:
# Cell 4：Gene expression预处理
# ============================================================

gene_pp = gene.copy()
gene_pp.var_names_make_unique()

gene_counts, gene_count_source = get_count_matrix(
    gene_pp,
    COUNT_LAYER,
)
gene_pp.X = gene_counts.copy()
gene_pp.layers["counts"] = gene_counts.copy()

sc.pp.filter_genes(
    gene_pp,
    min_cells=GENE_MIN_SPOTS,
)

n_top_gene = min(
    N_TOP_GENES,
    gene_pp.n_vars,
)

sc.pp.highly_variable_genes(
    gene_pp,
    layer="counts",
    flavor="seurat_v3",
    n_top_genes=n_top_gene,
)

sc.pp.normalize_total(
    gene_pp,
    target_sum=1e4,
)
sc.pp.log1p(gene_pp)

gene_high = gene_pp[
    :,
    gene_pp.var["highly_variable"],
].copy()

sc.pp.scale(
    gene_high,
    max_value=10,
)

n_pcs_gene = min(
    N_PCS,
    gene_high.n_obs - 1,
    gene_high.n_vars - 1,
)

gene_pp.obsm["feat"] = pca(
    gene_high,
    n_comps=n_pcs_gene,
).astype(np.float32)

print("Gene预处理完成")
print("Counts来源:", gene_count_source)
print("Gene矩阵:  ", gene_pp.shape)
print("HVG数量:   ", int(gene_pp.var["highly_variable"].sum()))
print("feat:      ", gene_pp.obsm["feat"].shape)


In [ ]:
# Cell 5：PSI预处理
# ============================================================

psi_matrix = sparse.csr_matrix(
    psi.X,
    dtype=np.float32,
)
observed_matrix = sparse.csr_matrix(
    psi.layers["observed"],
    dtype=np.float32,
)

n_observed_spots = np.asarray(
    observed_matrix.sum(axis=0)
).ravel()

min_observed_spots = max(
    2,
    int(np.ceil(PSI_MIN_OBS_FRAC * psi.n_obs)),
)

candidate_idx = np.flatnonzero(
    n_observed_spots >= min_observed_spots
)

if len(candidate_idx) < 2:
    raise ValueError("PSI覆盖过滤后事件不足")


# arcsin-sqrt变换，保留真实0和1
psi_transformed = psi_matrix.copy()
psi_transformed.data = np.arcsin(
    np.sqrt(
        np.clip(psi_transformed.data, 0, 1)
    )
).astype(np.float32)


# 只在PSI可观测的spot中计算事件均值和方差
feature_sum = np.asarray(
    psi_transformed.sum(axis=0)
).ravel()

feature_square_sum = np.asarray(
    psi_transformed
    .multiply(psi_transformed)
    .sum(axis=0)
).ravel()

feature_mean = np.divide(
    feature_sum,
    n_observed_spots,
    out=np.zeros_like(feature_sum),
    where=n_observed_spots > 0,
)

feature_variance = np.divide(
    feature_square_sum,
    n_observed_spots,
    out=np.zeros_like(feature_square_sum),
    where=n_observed_spots > 0,
) - feature_mean**2

feature_variance = np.maximum(
    feature_variance,
    0,
)

# 排除完全没有变化的事件
candidate_idx = candidate_idx[
    feature_variance[candidate_idx] > 0
]

n_top_psi = min(
    N_TOP_PSI,
    len(candidate_idx),
)

selected_idx = candidate_idx[
    np.argsort(
        feature_variance[candidate_idx]
    )[::-1][:n_top_psi]
]

psi_pp = psi[:, selected_idx].copy()


# PCA前构建dense矩阵
psi_dense = (
    psi_transformed[:, selected_idx]
    .toarray()
    .astype(np.float32)
)

observed_dense = (
    observed_matrix[:, selected_idx]
    .toarray()
    .astype(bool)
)

selected_mean = feature_mean[
    selected_idx
].astype(np.float32)

missing_mask = ~observed_dense

# 只填补未检测位置；真实PSI=0保持0
psi_dense[missing_mask] = np.broadcast_to(
    selected_mean,
    psi_dense.shape,
)[missing_mask]

psi_scaled = StandardScaler().fit_transform(
    psi_dense
).astype(np.float32)

n_pcs_psi = min(
    N_PCS,
    psi_scaled.shape[0] - 1,
    psi_scaled.shape[1] - 1,
)

psi_pp.obsm["feat"] = PCA(
    n_components=n_pcs_psi,
    random_state=RANDOM_SEED,
).fit_transform(
    psi_scaled
).astype(np.float32)

psi_pp.var["n_observed_spots"] = (
    n_observed_spots[selected_idx]
)
psi_pp.var["observed_fraction"] = (
    n_observed_spots[selected_idx] / psi.n_obs
)
psi_pp.var["psi_variance"] = (
    feature_variance[selected_idx]
)
psi_pp.var["highly_variable"] = True


# 两模态使用相同PCA维度
common_dim = min(
    gene_pp.obsm["feat"].shape[1],
    psi_pp.obsm["feat"].shape[1],
)
gene_pp.obsm["feat"] = gene_pp.obsm["feat"][:, :common_dim]
psi_pp.obsm["feat"] = psi_pp.obsm["feat"][:, :common_dim]

PSI_SELECTED_PATH = (
    OUT_DIR / f"{SAMPLE_NAME}_PSI_selected.h5ad"
)
psi_pp.write_h5ad(
    PSI_SELECTED_PATH,
    compression="gzip",
)

print("PSI预处理完成")
print("过滤前事件数:    ", psi.n_vars)
print("最低覆盖spot数:  ", min_observed_spots)
print("候选事件数:      ", len(candidate_idx))
print("最终选择事件数:  ", psi_pp.n_vars)
print("缺失填补比例:    ", round(missing_mask.mean(), 4))
print("Gene feat:       ", gene_pp.obsm["feat"].shape)
print("PSI feat:        ", psi_pp.obsm["feat"].shape)


In [ ]:
# Cell 6：构建空间图和特征图
# ============================================================

assert np.array_equal(
    gene_pp.obs_names,
    psi_pp.obs_names,
)

assert (
    gene_pp.obsm["feat"].shape[1]
    == psi_pp.obsm["feat"].shape[1]
)

data = construct_neighbor_graph(
    gene_pp,
    psi_pp,
    datatype=DATA_TYPE,
    n_neighbors=SPATIAL_NEIGHBORS,
)

print("邻接图构建完成")


In [ ]:
# Cell 7：训练SpatialGlue
# ============================================================

model = Train_SpatialGlue(
    data,
    datatype=DATA_TYPE,
    device=device,
    random_seed=RANDOM_SEED,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    dim_output=LATENT_DIM,
    weight_factors=WEIGHT_FACTORS,
)

# 官方代码会按datatype覆盖这两个参数，因此重新赋值
model.epochs = EPOCHS
model.weight_factors = list(WEIGHT_FACTORS)

output = model.train()

print("SpatialGlue训练完成")
print("联合表示:", output["SpatialGlue"].shape)


In [ ]:
# Cell 8：整理结果并聚类
# ============================================================

adata_sg = gene_pp.copy()

adata_sg.obsm["emb_latent_gene"] = (
    output["emb_latent_omics1"].copy()
)
adata_sg.obsm["emb_latent_PSI"] = (
    output["emb_latent_omics2"].copy()
)
adata_sg.obsm["SpatialGlue"] = (
    output["SpatialGlue"].copy()
)
adata_sg.obsm["alpha"] = output["alpha"].copy()
adata_sg.obsm["alpha_omics1"] = (
    output["alpha_omics1"].copy()
)
adata_sg.obsm["alpha_omics2"] = (
    output["alpha_omics2"].copy()
)

# alpha才是Gene与PSI之间的权重
adata_sg.obs["weight_gene"] = output["alpha"][:, 0]
adata_sg.obs["weight_PSI"] = output["alpha"][:, 1]

clustering(
    adata_sg,
    key="SpatialGlue",
    add_key=CLUSTER_KEY,
    n_clusters=N_CLUSTERS,
    method=CLUSTER_METHOD,
    use_pca=True,
)

print("聚类完成")
print(
    adata_sg.obs[CLUSTER_KEY]
    .value_counts()
    .sort_index()
)


In [ ]:
# Cell 9：UMAP和空间聚类图
# ============================================================

sc.pp.neighbors(
    adata_sg,
    use_rep="SpatialGlue",
    n_neighbors=30,
)
sc.tl.umap(
    adata_sg,
    random_state=RANDOM_SEED,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
)

sc.pl.umap(
    adata_sg,
    color=CLUSTER_KEY,
    ax=axes[0],
    show=False,
    title="SpatialGlue UMAP",
)

sc.pl.embedding(
    adata_sg,
    basis="spatial",
    color=CLUSTER_KEY,
    size=70,
    ax=axes[1],
    show=False,
    title="SpatialGlue spatial",
)

plt.tight_layout()
fig.savefig(
    FIG_DIR / f"{SAMPLE_NAME}_clusters.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


In [ ]:
# Cell 10：Gene与PSI模态权重
# ============================================================

print(
    "Gene平均权重:",
    round(adata_sg.obs["weight_gene"].mean(), 4),
)
print(
    "PSI平均权重:",
    round(adata_sg.obs["weight_PSI"].mean(), 4),
)

weight_sum = (
    adata_sg.obs["weight_gene"]
    + adata_sg.obs["weight_PSI"]
)
print(
    "每个spot两权重和范围:",
    round(weight_sum.min(), 6),
    "–",
    round(weight_sum.max(), 6),
)


# 空间权重图
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
)

sc.pl.embedding(
    adata_sg,
    basis="spatial",
    color="weight_gene",
    cmap="Reds",
    size=25,
    ax=axes[0],
    show=False,
    title="Gene modality weight",
)

sc.pl.embedding(
    adata_sg,
    basis="spatial",
    color="weight_PSI",
    cmap="Blues",
    size=25,
    ax=axes[1],
    show=False,
    title="PSI modality weight",
)

plt.tight_layout()
fig.savefig(
    FIG_DIR / f"{SAMPLE_NAME}_weights.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# 按Leiden簇汇总：均值±SEM
weight_df = adata_sg.obs[
    [CLUSTER_KEY, "weight_gene", "weight_PSI"]
].copy()
weight_df[CLUSTER_KEY] = weight_df[CLUSTER_KEY].astype(str)


def cluster_sort_key(label):
    try:
        return (0, float(label))
    except ValueError:
        return (1, str(label))


cluster_order = sorted(
    weight_df[CLUSTER_KEY].unique(),
    key=cluster_sort_key,
)

weight_summary = (
    weight_df
    .groupby(CLUSTER_KEY, observed=True)
    .agg(
        n_spots=("weight_gene", "size"),
        gene_mean=("weight_gene", "mean"),
        gene_sem=("weight_gene", "sem"),
        psi_mean=("weight_PSI", "mean"),
        psi_sem=("weight_PSI", "sem"),
    )
    .reindex(cluster_order)
)

weight_summary[["gene_sem", "psi_sem"]] = (
    weight_summary[["gene_sem", "psi_sem"]]
    .fillna(0)
)

display(weight_summary.round(4))
weight_summary.to_csv(
    OUT_DIR / f"{SAMPLE_NAME}_weights_by_leiden.csv"
)

x = np.arange(len(weight_summary))
width = 0.38

fig, ax = plt.subplots(
    figsize=(max(8, 0.8 * len(weight_summary)), 5)
)

ax.bar(
    x - width / 2,
    weight_summary["gene_mean"],
    width=width,
    yerr=weight_summary["gene_sem"],
    capsize=3,
    color="#D95F5F",
    label="Gene",
)
ax.bar(
    x + width / 2,
    weight_summary["psi_mean"],
    width=width,
    yerr=weight_summary["psi_sem"],
    capsize=3,
    color="#4C78A8",
    label="PSI",
)

ax.set_xticks(x)
ax.set_xticklabels([
    f"{cluster}\n(n={int(n)})"
    for cluster, n in zip(
        weight_summary.index,
        weight_summary["n_spots"],
    )
])
ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Mean modality weight ± SEM")
ax.set_ylim(0, 1.05)
ax.set_title("Modality weights across Leiden clusters")
ax.legend(frameon=False)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
fig.savefig(
    FIG_DIR / f"{SAMPLE_NAME}_weights_by_leiden.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


In [ ]:
# Cell 11：保存SpatialGlue结果
# ============================================================

adata_sg.uns["spatialglue_gene_PSI"] = {
    "sample_name": SAMPLE_NAME,
    "gene_path": str(GENE_PATH),
    "inclusion_path": str(INCLUSION_PATH),
    "skipping_path": str(SKIPPING_PATH),
    "psi_min_total": PSI_MIN_TOTAL,
    "psi_min_obs_frac": PSI_MIN_OBS_FRAC,
    "rmats_significant_filter": USE_RMATS_SIG_FILTER,
    "n_top_genes": N_TOP_GENES,
    "n_top_psi": N_TOP_PSI,
    "n_pcs": int(gene_pp.obsm["feat"].shape[1]),
    "weight_factors": list(WEIGHT_FACTORS),
    "random_seed": RANDOM_SEED,
    "n_clusters": N_CLUSTERS,
}

OUT_PATH = (
    OUT_DIR / f"{SAMPLE_NAME}_SpatialGlue_gene_PSI.h5ad"
)

adata_sg.write_h5ad(
    OUT_PATH,
    compression="gzip",
)

print("保存完成")
print("PSI矩阵:       ", PSI_PATH)
print("筛选后PSI矩阵: ", PSI_SELECTED_PATH)
print("SpatialGlue:   ", OUT_PATH)


# 肾

In [ ]:
# Cell 1：导入、路径和参数
# ============================================================

import os
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch

from scipy import sparse
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# ------------------------- 样本选择 -------------------------

# 只改这一行："G24-2" 或 "G25-6"
SAMPLE_KEY = "G25-6"

BASE_DIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie"
)

SAMPLE_CONFIG = {
    "G24-2": {
        "sample_name": "G24-2_Cerebellum",
        "gene_path": (
            BASE_DIR
            / "20260210_2/0423_repaired_data/G24-2_repaired.h5ad"
        ),
        "psi_dir": (
            BASE_DIR
            / "20260427_AS_analysis/Fig4_cluster_psi/results"
            / "G24-2_Cerebellum"
        ),
    },
    "G25-6": {
        "sample_name": "G25-6_Kidney",
        "gene_path": (
            BASE_DIR
            / "20260210_2/0423_repaired_data/G25-6_repaired.h5ad"
        ),
        "psi_dir": (
            BASE_DIR
            / "20260427_AS_analysis/Fig4_cluster_psi/results"
            / "G25-6_Kidney"
        ),
    },
}

if SAMPLE_KEY not in SAMPLE_CONFIG:
    raise KeyError(
        f"未知SAMPLE_KEY：{SAMPLE_KEY}"
    )

cfg = SAMPLE_CONFIG[SAMPLE_KEY]

SAMPLE_NAME = cfg["sample_name"]
GENE_PATH = cfg["gene_path"]
INCLUSION_PATH = cfg["psi_dir"] / "inc.h5ad"
SKIPPING_PATH = cfg["psi_dir"] / "ski.h5ad"
OUT_DIR = cfg["psi_dir"] / "SpatialGlue_gene_PSI"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------- PSI构建参数 -------------------------

IN_TISSUE_ONLY = True

# 优先读取layers["counts"]；不存在时读取X
COUNT_LAYER = "counts"

# spot-event至少需要多少个I+S计数才认为PSI可观测
PSI_MIN_TOTAL = 1

# 是否沿用旧脚本的rMATS显著事件筛选
# 无监督聚类建议False，避免把已有分组信息带入聚类
USE_RMATS_SIG_FILTER = False
FDR_THRESH = 0.05
DPSI_THRESH = 0.05


# ------------------------- 预处理参数 -------------------------

GENE_MIN_SPOTS = 5
N_TOP_GENES = 3000

# 一个事件至少在1%的spot中有I+S计数
PSI_MIN_OBS_FRAC = 0.01
N_TOP_PSI = 3000
N_PCS = 50


# ------------------------- SpatialGlue参数 -------------------------

RANDOM_SEED = 2022
DATA_TYPE = "10x"
SPATIAL_NEIGHBORS = 3

EPOCHS = 200
LEARNING_RATE = 1e-4
LATENT_DIM = 64

# [gene重建, PSI重建, gene对应, PSI对应]
# 等权设置避免预先偏向PSI，适合评价PSI是否提供额外信息
WEIGHT_FACTORS = [1, 1, 1, 1]

CLUSTER_METHOD = "leiden"
N_CLUSTERS = 10
CLUSTER_KEY = "SpatialGlue_cluster"


# ------------------------- 环境和绘图 -------------------------

R_HOME = "/histor/zhao/lupeng/micromamba/envs/SpatialGlue/lib/R"
os.environ["R_HOME"] = R_HOME

from SpatialGlue.preprocess import (
    construct_neighbor_graph,
    fix_seed,
    pca,
)
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue
from SpatialGlue.utils import clustering

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
sc.settings._vector_friendly = True

fix_seed(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", device)
print("Sample:", SAMPLE_NAME)
print("Gene:", GENE_PATH)
print("Inclusion:", INCLUSION_PATH)
print("Skipping:", SKIPPING_PATH)
print("Output:", OUT_DIR)


In [ ]:
# Cell 2：读取数据并对齐spot和事件
# ============================================================


def get_count_matrix(adata, layer="counts"):
    """优先读取原始counts层；不存在时使用X。"""
    if layer is not None and layer in adata.layers:
        matrix = adata.layers[layer]
        source = f'layers["{layer}"]'
    else:
        matrix = adata.X
        source = "X"

    matrix = sparse.csr_matrix(
        matrix,
        dtype=np.float32,
    )

    if matrix.nnz > 0 and matrix.data.min() < 0:
        raise ValueError("计数矩阵不能包含负值")

    return matrix, source


gene = sc.read_h5ad(GENE_PATH)
inc = ad.read_h5ad(INCLUSION_PATH)
ski = ad.read_h5ad(SKIPPING_PATH)

print("原始矩阵")
print("Gene:      ", gene.shape)
print("Inclusion: ", inc.shape)
print("Skipping:  ", ski.shape)


# 只保留组织内spot
if IN_TISSUE_ONLY and "in_tissue" in gene.obs:
    keep = (
        gene.obs["in_tissue"]
        .astype(str)
        .str.lower()
        .isin(["1", "true"])
    )
    gene = gene[keep].copy()


# 按Gene矩阵顺序对齐spot
common_spots = gene.obs_names[
    gene.obs_names.isin(inc.obs_names)
    & gene.obs_names.isin(ski.obs_names)
]

if len(common_spots) == 0:
    raise ValueError("三个矩阵没有共同spot")

gene = gene[common_spots].copy()
inc = inc[common_spots].copy()
ski = ski[common_spots].copy()


# 按Inclusion矩阵顺序对齐事件
common_events = inc.var_names[
    inc.var_names.isin(ski.var_names)
]

if len(common_events) == 0:
    raise ValueError("Inclusion与Skipping没有共同事件")

inc = inc[:, common_events].copy()
ski = ski[:, common_events].copy()

assert np.array_equal(inc.obs_names, ski.obs_names)
assert np.array_equal(inc.var_names, ski.var_names)

if "spatial" not in gene.obsm:
    raise KeyError('Gene矩阵缺少obsm["spatial"]')

inc_counts, inc_source = get_count_matrix(inc, COUNT_LAYER)
ski_counts, ski_source = get_count_matrix(ski, COUNT_LAYER)

print("\n对齐后")
print("Spots: ", len(common_spots))
print("Events:", len(common_events))
print("Inclusion counts来源:", inc_source)
print("Skipping counts来源: ", ski_source)


## 构建psi

In [ ]:
# Cell 3：构建并保存PSI矩阵
# ============================================================

event_var = inc.var.copy()


# 可选：沿用旧脚本的FDR和dPSI筛选
if USE_RMATS_SIG_FILTER:
    required = ["FDR", "dPSI"]
    missing = [x for x in required if x not in event_var]
    if missing:
        raise KeyError(f"事件信息缺少列：{missing}")

    fdr = pd.to_numeric(
        event_var["FDR"],
        errors="coerce",
    )
    dpsi = pd.to_numeric(
        event_var["dPSI"],
        errors="coerce",
    )

    keep_event = (
        fdr.notna()
        & (fdr <= FDR_THRESH)
        & dpsi.notna()
        & (dpsi.abs() > DPSI_THRESH)
    ).to_numpy()

    inc_counts = inc_counts[:, keep_event]
    ski_counts = ski_counts[:, keep_event]
    event_var = event_var.iloc[keep_event].copy()

    print("rMATS显著筛选后事件数:", keep_event.sum())


# 必须使用原始I/S计数计算，不对两个矩阵分别normalize_total
total_counts = (
    inc_counts + ski_counts
).tocsr()

# observed=1：该spot-event的I+S达到最低计数
observed = total_counts.copy()
observed.data = (
    observed.data >= PSI_MIN_TOTAL
).astype(np.uint8)
observed.eliminate_zeros()

# 稀疏计算 PSI = I/(I+S)
inverse_total = total_counts.copy().astype(np.float32)
valid_data = inverse_total.data >= PSI_MIN_TOTAL
inverse_total.data = np.where(
    valid_data,
    1.0 / inverse_total.data,
    0.0,
).astype(np.float32)
inverse_total.eliminate_zeros()

psi_matrix = (
    inc_counts.multiply(inverse_total)
).tocsr().astype(np.float32)
psi_matrix.eliminate_zeros()

if psi_matrix.nnz > 0:
    if psi_matrix.data.min() < -1e-7 or psi_matrix.data.max() > 1 + 1e-7:
        raise ValueError("计算得到的PSI不在0–1范围")


psi = ad.AnnData(
    X=psi_matrix,
    obs=gene.obs.copy(),
    var=event_var.copy(),
)

psi.layers["inclusion_counts"] = inc_counts.astype(np.float32)
psi.layers["skipping_counts"] = ski_counts.astype(np.float32)
psi.layers["total_counts"] = total_counts.astype(np.float32)
psi.layers["observed"] = observed

psi.obsm["spatial"] = gene.obsm["spatial"].copy()
if "spatial" in gene.uns:
    psi.uns["spatial"] = gene.uns["spatial"].copy()

psi.uns["psi_definition"] = {
    "formula": "inclusion_counts / (inclusion_counts + skipping_counts)",
    "count_source_inclusion": inc_source,
    "count_source_skipping": ski_source,
    "min_total_per_spot_event": PSI_MIN_TOTAL,
    "missing_definition": "I + S < min_total",
    "missing_not_equal_zero": True,
    "rmats_significant_filter": USE_RMATS_SIG_FILTER,
}

PSI_PATH = OUT_DIR / f"{SAMPLE_NAME}_PSI.h5ad"
psi.write_h5ad(PSI_PATH, compression="gzip")

n_all = psi.n_obs * psi.n_vars
n_observed = observed.nnz
n_true_zero = observed.nnz - psi_matrix.nnz

print("PSI矩阵构建完成")
print("Shape:            ", psi.shape)
print("Observed比例:     ", round(n_observed / n_all, 4))
print("缺失比例:         ", round(1 - n_observed / n_all, 4))
print("真实PSI=0位置数:  ", f"{n_true_zero:,}")
print("保存路径:         ", PSI_PATH)


## 预处理

In [ ]:
# Cell 4：Gene expression预处理
# ============================================================

gene_pp = gene.copy()
gene_pp.var_names_make_unique()

gene_counts, gene_count_source = get_count_matrix(
    gene_pp,
    COUNT_LAYER,
)
gene_pp.X = gene_counts.copy()
gene_pp.layers["counts"] = gene_counts.copy()

sc.pp.filter_genes(
    gene_pp,
    min_cells=GENE_MIN_SPOTS,
)

n_top_gene = min(
    N_TOP_GENES,
    gene_pp.n_vars,
)

sc.pp.highly_variable_genes(
    gene_pp,
    layer="counts",
    flavor="seurat_v3",
    n_top_genes=n_top_gene,
)

sc.pp.normalize_total(
    gene_pp,
    target_sum=1e4,
)
sc.pp.log1p(gene_pp)

gene_high = gene_pp[
    :,
    gene_pp.var["highly_variable"],
].copy()

sc.pp.scale(
    gene_high,
    max_value=10,
)

n_pcs_gene = min(
    N_PCS,
    gene_high.n_obs - 1,
    gene_high.n_vars - 1,
)

gene_pp.obsm["feat"] = pca(
    gene_high,
    n_comps=n_pcs_gene,
).astype(np.float32)

print("Gene预处理完成")
print("Counts来源:", gene_count_source)
print("Gene矩阵:  ", gene_pp.shape)
print("HVG数量:   ", int(gene_pp.var["highly_variable"].sum()))
print("feat:      ", gene_pp.obsm["feat"].shape)


In [ ]:
# Cell 5：PSI预处理
# ============================================================

psi_matrix = sparse.csr_matrix(
    psi.X,
    dtype=np.float32,
)
observed_matrix = sparse.csr_matrix(
    psi.layers["observed"],
    dtype=np.float32,
)

n_observed_spots = np.asarray(
    observed_matrix.sum(axis=0)
).ravel()

min_observed_spots = max(
    2,
    int(np.ceil(PSI_MIN_OBS_FRAC * psi.n_obs)),
)

candidate_idx = np.flatnonzero(
    n_observed_spots >= min_observed_spots
)

if len(candidate_idx) < 2:
    raise ValueError("PSI覆盖过滤后事件不足")


# arcsin-sqrt变换，保留真实0和1
psi_transformed = psi_matrix.copy()
psi_transformed.data = np.arcsin(
    np.sqrt(
        np.clip(psi_transformed.data, 0, 1)
    )
).astype(np.float32)


# 只在PSI可观测的spot中计算事件均值和方差
feature_sum = np.asarray(
    psi_transformed.sum(axis=0)
).ravel()

feature_square_sum = np.asarray(
    psi_transformed
    .multiply(psi_transformed)
    .sum(axis=0)
).ravel()

feature_mean = np.divide(
    feature_sum,
    n_observed_spots,
    out=np.zeros_like(feature_sum),
    where=n_observed_spots > 0,
)

feature_variance = np.divide(
    feature_square_sum,
    n_observed_spots,
    out=np.zeros_like(feature_square_sum),
    where=n_observed_spots > 0,
) - feature_mean**2

feature_variance = np.maximum(
    feature_variance,
    0,
)

# 排除完全没有变化的事件
candidate_idx = candidate_idx[
    feature_variance[candidate_idx] > 0
]

n_top_psi = min(
    N_TOP_PSI,
    len(candidate_idx),
)

selected_idx = candidate_idx[
    np.argsort(
        feature_variance[candidate_idx]
    )[::-1][:n_top_psi]
]

psi_pp = psi[:, selected_idx].copy()


# PCA前构建dense矩阵
psi_dense = (
    psi_transformed[:, selected_idx]
    .toarray()
    .astype(np.float32)
)

observed_dense = (
    observed_matrix[:, selected_idx]
    .toarray()
    .astype(bool)
)

selected_mean = feature_mean[
    selected_idx
].astype(np.float32)

missing_mask = ~observed_dense

# 只填补未检测位置；真实PSI=0保持0
psi_dense[missing_mask] = np.broadcast_to(
    selected_mean,
    psi_dense.shape,
)[missing_mask]

psi_scaled = StandardScaler().fit_transform(
    psi_dense
).astype(np.float32)

n_pcs_psi = min(
    N_PCS,
    psi_scaled.shape[0] - 1,
    psi_scaled.shape[1] - 1,
)

psi_pp.obsm["feat"] = PCA(
    n_components=n_pcs_psi,
    random_state=RANDOM_SEED,
).fit_transform(
    psi_scaled
).astype(np.float32)

psi_pp.var["n_observed_spots"] = (
    n_observed_spots[selected_idx]
)
psi_pp.var["observed_fraction"] = (
    n_observed_spots[selected_idx] / psi.n_obs
)
psi_pp.var["psi_variance"] = (
    feature_variance[selected_idx]
)
psi_pp.var["highly_variable"] = True


# 两模态使用相同PCA维度
common_dim = min(
    gene_pp.obsm["feat"].shape[1],
    psi_pp.obsm["feat"].shape[1],
)
gene_pp.obsm["feat"] = gene_pp.obsm["feat"][:, :common_dim]
psi_pp.obsm["feat"] = psi_pp.obsm["feat"][:, :common_dim]

PSI_SELECTED_PATH = (
    OUT_DIR / f"{SAMPLE_NAME}_PSI_selected.h5ad"
)
psi_pp.write_h5ad(
    PSI_SELECTED_PATH,
    compression="gzip",
)

print("PSI预处理完成")
print("过滤前事件数:    ", psi.n_vars)
print("最低覆盖spot数:  ", min_observed_spots)
print("候选事件数:      ", len(candidate_idx))
print("最终选择事件数:  ", psi_pp.n_vars)
print("缺失填补比例:    ", round(missing_mask.mean(), 4))
print("Gene feat:       ", gene_pp.obsm["feat"].shape)
print("PSI feat:        ", psi_pp.obsm["feat"].shape)


In [ ]:
# Cell 6：构建空间图和特征图
# ============================================================

assert np.array_equal(
    gene_pp.obs_names,
    psi_pp.obs_names,
)

assert (
    gene_pp.obsm["feat"].shape[1]
    == psi_pp.obsm["feat"].shape[1]
)

data = construct_neighbor_graph(
    gene_pp,
    psi_pp,
    datatype=DATA_TYPE,
    n_neighbors=SPATIAL_NEIGHBORS,
)

print("邻接图构建完成")


## 训练SG

In [ ]:
# Cell 7：训练SpatialGlue
# ============================================================

model = Train_SpatialGlue(
    data,
    datatype=DATA_TYPE,
    device=device,
    random_seed=RANDOM_SEED,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    dim_output=LATENT_DIM,
    weight_factors=WEIGHT_FACTORS,
)

# 官方代码会按datatype覆盖这两个参数，因此重新赋值
model.epochs = EPOCHS
model.weight_factors = list(WEIGHT_FACTORS)

output = model.train()

print("SpatialGlue训练完成")
print("联合表示:", output["SpatialGlue"].shape)


In [ ]:
# Cell 8：整理结果并聚类
# ============================================================

adata_sg = gene_pp.copy()

adata_sg.obsm["emb_latent_gene"] = (
    output["emb_latent_omics1"].copy()
)
adata_sg.obsm["emb_latent_PSI"] = (
    output["emb_latent_omics2"].copy()
)
adata_sg.obsm["SpatialGlue"] = (
    output["SpatialGlue"].copy()
)
adata_sg.obsm["alpha"] = output["alpha"].copy()
adata_sg.obsm["alpha_omics1"] = (
    output["alpha_omics1"].copy()
)
adata_sg.obsm["alpha_omics2"] = (
    output["alpha_omics2"].copy()
)

# alpha才是Gene与PSI之间的权重
adata_sg.obs["weight_gene"] = output["alpha"][:, 0]
adata_sg.obs["weight_PSI"] = output["alpha"][:, 1]

clustering(
    adata_sg,
    key="SpatialGlue",
    add_key=CLUSTER_KEY,
    n_clusters=N_CLUSTERS,
    method=CLUSTER_METHOD,
    use_pca=True,
)

print("聚类完成")
print(
    adata_sg.obs[CLUSTER_KEY]
    .value_counts()
    .sort_index()
)


## 可视化

In [ ]:
# Cell 9：UMAP和空间聚类图
# ============================================================

sc.pp.neighbors(
    adata_sg,
    use_rep="SpatialGlue",
    n_neighbors=30,
)
sc.tl.umap(
    adata_sg,
    random_state=RANDOM_SEED,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
)

sc.pl.umap(
    adata_sg,
    color=CLUSTER_KEY,
    ax=axes[0],
    show=False,
    title="SpatialGlue UMAP",
)

sc.pl.embedding(
    adata_sg,
    basis="spatial",
    color=CLUSTER_KEY,
    size=70,
    ax=axes[1],
    show=False,
    title="SpatialGlue spatial",
)

plt.tight_layout()
fig.savefig(
    FIG_DIR / f"{SAMPLE_NAME}_clusters.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


In [ ]:
# Cell 10：Gene与PSI模态权重
# ============================================================

print(
    "Gene平均权重:",
    round(adata_sg.obs["weight_gene"].mean(), 4),
)
print(
    "PSI平均权重:",
    round(adata_sg.obs["weight_PSI"].mean(), 4),
)

weight_sum = (
    adata_sg.obs["weight_gene"]
    + adata_sg.obs["weight_PSI"]
)
print(
    "每个spot两权重和范围:",
    round(weight_sum.min(), 6),
    "–",
    round(weight_sum.max(), 6),
)


# 空间权重图
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
)

sc.pl.embedding(
    adata_sg,
    basis="spatial",
    color="weight_gene",
    cmap="Reds",
    size=25,
    ax=axes[0],
    show=False,
    title="Gene modality weight",
)

sc.pl.embedding(
    adata_sg,
    basis="spatial",
    color="weight_PSI",
    cmap="Blues",
    size=25,
    ax=axes[1],
    show=False,
    title="PSI modality weight",
)

plt.tight_layout()
fig.savefig(
    FIG_DIR / f"{SAMPLE_NAME}_weights.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# 按Leiden簇汇总：均值±SEM
weight_df = adata_sg.obs[
    [CLUSTER_KEY, "weight_gene", "weight_PSI"]
].copy()
weight_df[CLUSTER_KEY] = weight_df[CLUSTER_KEY].astype(str)


def cluster_sort_key(label):
    try:
        return (0, float(label))
    except ValueError:
        return (1, str(label))


cluster_order = sorted(
    weight_df[CLUSTER_KEY].unique(),
    key=cluster_sort_key,
)

weight_summary = (
    weight_df
    .groupby(CLUSTER_KEY, observed=True)
    .agg(
        n_spots=("weight_gene", "size"),
        gene_mean=("weight_gene", "mean"),
        gene_sem=("weight_gene", "sem"),
        psi_mean=("weight_PSI", "mean"),
        psi_sem=("weight_PSI", "sem"),
    )
    .reindex(cluster_order)
)

weight_summary[["gene_sem", "psi_sem"]] = (
    weight_summary[["gene_sem", "psi_sem"]]
    .fillna(0)
)

display(weight_summary.round(4))
weight_summary.to_csv(
    OUT_DIR / f"{SAMPLE_NAME}_weights_by_leiden.csv"
)

x = np.arange(len(weight_summary))
width = 0.38

fig, ax = plt.subplots(
    figsize=(max(8, 0.8 * len(weight_summary)), 5)
)

ax.bar(
    x - width / 2,
    weight_summary["gene_mean"],
    width=width,
    yerr=weight_summary["gene_sem"],
    capsize=3,
    color="#D95F5F",
    label="Gene",
)
ax.bar(
    x + width / 2,
    weight_summary["psi_mean"],
    width=width,
    yerr=weight_summary["psi_sem"],
    capsize=3,
    color="#4C78A8",
    label="PSI",
)

ax.set_xticks(x)
ax.set_xticklabels([
    f"{cluster}\n(n={int(n)})"
    for cluster, n in zip(
        weight_summary.index,
        weight_summary["n_spots"],
    )
])
ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Mean modality weight ± SEM")
ax.set_ylim(0, 1.05)
ax.set_title("Modality weights across Leiden clusters")
ax.legend(frameon=False)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
fig.savefig(
    FIG_DIR / f"{SAMPLE_NAME}_weights_by_leiden.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


## 保存

In [ ]:
# Cell 11：保存SpatialGlue结果
# ============================================================

adata_sg.uns["spatialglue_gene_PSI"] = {
    "sample_name": SAMPLE_NAME,
    "gene_path": str(GENE_PATH),
    "inclusion_path": str(INCLUSION_PATH),
    "skipping_path": str(SKIPPING_PATH),
    "psi_min_total": PSI_MIN_TOTAL,
    "psi_min_obs_frac": PSI_MIN_OBS_FRAC,
    "rmats_significant_filter": USE_RMATS_SIG_FILTER,
    "n_top_genes": N_TOP_GENES,
    "n_top_psi": N_TOP_PSI,
    "n_pcs": int(gene_pp.obsm["feat"].shape[1]),
    "weight_factors": list(WEIGHT_FACTORS),
    "random_seed": RANDOM_SEED,
    "n_clusters": N_CLUSTERS,
}

OUT_PATH = (
    OUT_DIR / f"{SAMPLE_NAME}_SpatialGlue_gene_PSI.h5ad"
)

adata_sg.write_h5ad(
    OUT_PATH,
    compression="gzip",
)

print("保存完成")
print("PSI矩阵:       ", PSI_PATH)
print("筛选后PSI矩阵: ", PSI_SELECTED_PATH)
print("SpatialGlue:   ", OUT_PATH)


# 0000